# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.5 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

In this assignment, I used `wc -l` in the terminal to count the CSV lines and wrote a simple `profile` function with a loop to check the shapes and duplicates of the raw files. For the Lecture 03 NumPy part, I chose `axis=0` because each column is a visit (`baseline` and `week12`), so `axis=0` averages down the rows across all patients to give one mean per visit. Finally, I cleaned the missing codes and text labels, melted the wide A1c table, and merged all the tables together with `validate` to make sure the keys matched properly.


## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt
cat output/file_inventory.txt

       6 data/sites.csv
   26298 data/visits_north.csv
   31075 data/visits_south.csv
   57379 total


### 1.2 Profile each table with a function

In [3]:
def profile(name, table):
    """Return a summary dictionary with table name, rows, columns, and duplicate count."""
    return {
        "table": name,
        "rows": int(table.shape[0]),
        "columns": int(table.shape[1]),
        "exact_duplicate_rows": int(table.duplicated().sum()),}


results = [profile(name, raw[Path(name).stem]) for name in manifest["files"]]
table_profile = pd.DataFrame(results)
display(table_profile)

table_profile.to_csv(PROFILE_PATH, index=False)

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

After cleaning, patients_clean has one row per patient with key patient_id (raw data had exact duplicates). sites_current keeps only current sites with key site_code (raw data had repeated codes for retired sites). visits_clean stacks both clinics and keeps the latest recheck, so it has one row per patient visit with composite key (patient_id, visit). a1c_long also uses (patient_id, visit) as its key after reshaping from wide form, where the key was patient_id. For Task 4.4, joining visits_clean to patients_clean is many-to-one, joining to sites_current is many-to-one, and joining to a1c_long is one-to-one.

### 2.2 Stack the two visit files

In [4]:
north_copy = raw["visits_north"].copy(deep=True)
north_copy["source_file"] = "visits_north.csv"

south_copy = (raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"}))
south_copy["source_file"] = "visits_south.csv"

visits_raw = pd.concat([north_copy, south_copy], ignore_index=True)

print("visits_raw shape:", visits_raw.shape)
display(visits_raw.head())

visits_raw shape: (57371, 7)


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv


### 2.3 Audit the raw tables

In [5]:
p_df = raw["patients"]
s_df = raw["sites"]
v_df = visits_raw


g_str = p_df["gender"].fillna("").astype(str)
g_stripped = g_str.str.strip()
issue_gender_not_code = int(
    ((g_stripped != "") & (~g_str.isin(["F", "M", "X"]))).sum())
issue_gender_empty = int((g_stripped == "").sum())


by_str = p_df["birth_year"].fillna("").astype(str).str.strip()
issue_birth_9999 = int((by_str == "9999").sum())


sc_str = p_df["site_code"].fillna("").astype(str)
sc_norm = (
    sc_str.str.upper()
    .str.replace(" ", "", regex=False)
    .str.replace("-", "", regex=False))
issue_site_norm = int((sc_str != sc_norm).sum())


issue_site_repeat = int(s_df.duplicated(subset=["site_code"], keep=False).sum())

v_label_str = v_df["visit"].fillna("").astype(str)
issue_visit_label = int((~v_label_str.isin(["baseline", "week12"])).sum())


sbp_num = pd.to_numeric(
    v_df["sbp_mmhg"].astype(str).str.strip(), errors="coerce"
)
issue_sbp_999 = int((sbp_num == 999).sum())
issue_sbp_range = int(
    ((sbp_num != 999) & sbp_num.notna() & ((sbp_num < 70) | (sbp_num > 250))).sum()
)

vd_str = v_df["visit_date"].fillna("").astype(str)
exact_fmt = vd_str.str.match(r"^\d{4}-\d{2}-\d{2}$")
valid_cal = pd.to_datetime(vd_str, format="%Y-%m-%d", errors="coerce").notna()
issue_date_fail = int((~(exact_fmt & valid_cal)).sum())


pid_norm = (
    v_df["patient_id"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(" ", "", regex=False)
    .str.replace("-", "", regex=False))

v_low = v_label_str.str.strip().str.lower()
visit_mapped = np.where(
    v_low.str.contains("12"),
    "week12",
    np.where(
        v_low.str.contains("base|init|enroll|0"),
        "baseline",
        v_low,),)

pair_df = pd.DataFrame({"patient_id": pid_norm, "visit": visit_mapped})
issue_repeat_pairs = int(
    pair_df.duplicated(subset=["patient_id", "visit"], keep=False).sum())


issue_audit = pd.DataFrame(
    [
        ("patients: gender values that are not a code", issue_gender_not_code),
        ("patients: empty gender", issue_gender_empty),
        ("patients: birth_year sentinel 9999", issue_birth_9999),
        ("patients: site codes needing normalization", issue_site_norm),
        ("sites: rows with a repeated site_code", issue_site_repeat),
        (
            "visits: visit labels that are not baseline or week12",
            issue_visit_label,
        ),
        ("visits: sbp sentinel 999", issue_sbp_999),
        ("visits: sbp outside 70 through 250", issue_sbp_range),
        ("visits: date parse failures", issue_date_fail),
        (
            "visits: rows in repeated patient_id and visit pairs",
            issue_repeat_pairs,),],
    columns=["issue", "count"],)

display(issue_audit)
issue_audit.to_csv(AUDIT_PATH, index=False)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

,issue,count
0,patients: gender values that are not a code,6968
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1811
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 through 250,116
8,visits: date parse failures,169
9,visits: rows in repeated patient_id and visit ...,340


## Task 3: Clean

### 3.1 Clean the patients

In [6]:
display(raw["patients"]["gender"].value_counts(dropna=False))

patients_clean = raw["patients"].copy(deep=True).drop_duplicates(keep="first")

gender_map = {
    "f": "F",
    "female": "F",
    "woman": "F",
    "w": "F",
    "m": "M",
    "male": "M",
    "man": "M",
    "x": "X",
    "nb": "X",
    "nonbinary": "X",
    "non-binary": "X",}

patients_clean["gender"] = (
    patients_clean["gender"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(gender_map)
    .astype("category"))


by_num = pd.to_numeric(
    patients_clean["birth_year"].astype(str).str.strip(), errors="coerce")
patients_clean["birth_year"] = by_num.where(by_num != 9999).astype("Int64")


patients_clean["site_code"] = (
    patients_clean["site_code"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False))

patients_clean = patients_clean.reset_index(drop=True)
display(patients_clean.head())

gender
F             11248
M             10971
male           1039
female         1018
Male            622
f               622
Female          578
m               538
woman           400
                382
X               372
man             329
unknown         282
MALE            256
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              104
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]

,patient_id,gender,birth_year,site_code
0,P00001,F,1966,S02
1,P00002,F,<NA>,N01
2,P00003,F,1970,N01
3,P00004,X,<NA>,N01
4,P00005,M,1947,N01


In [7]:
print(patients_clean["gender"].cat.categories)

patients_summary = pd.DataFrame({
    "metric": [
        "patients",
        "birth_year missing",
        "birth_year min",
        "birth_year max",
        "site_code values",
        "patients at N01",
        "patients at S02",
        "patients at E04",],

    "value": [
        int(len(patients_clean)),
        int(patients_clean["birth_year"].isna().sum()),
        int(patients_clean["birth_year"].min()),
        int(patients_clean["birth_year"].max()),
        int(patients_clean["site_code"].nunique()),
        int((patients_clean["site_code"] == "N01").sum()),
        int((patients_clean["site_code"] == "S02").sum()),
        int((patients_clean["site_code"] == "E04").sum()),
    ],})
display(patients_summary)
patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)

gender_counts = (patients_clean["gender"].value_counts(dropna=False).reset_index())
gender_counts["code"] = gender_counts["gender"].cat.codes
display(gender_counts)
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

Index(['F', 'M', 'X'], dtype='str')


,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


,gender,count,code
0,F,14413,0
1,M,13955,1
2,NaN,863,-1
3,X,769,2


### 3.2 Clean the visits

In [8]:
visits_clean = visits_raw.copy(deep=True).drop_duplicates(keep="first")

display(visits_clean["visit"].value_counts(dropna=False))

visit_map = {
    "baseline": "baseline",
    "base": "baseline",
    "b": "baseline",
    "initial": "baseline",
    "enrollment": "baseline",
    "enroll": "baseline",
    "wk0": "baseline",
    "week0": "baseline",
    "week 0": "baseline",
    "0": "baseline",
    "week12": "week12",
    "wk12": "week12",
    "week 12": "week12",
    "wk 12": "week12",
    "w12": "week12",
    "12w": "week12",
    "12wk": "week12",
    "12": "week12",
    "followup": "week12",
    "follow-up": "week12",}
for val in (visits_clean["visit"].dropna().astype(str).str.strip().str.lower().unique()):
    if val not in visit_map:
        visit_map[val] = "week12" if "12" in val else "baseline"

visits_clean["visit"] = (visits_clean["visit"].astype(str).str.strip().str.lower().map(visit_map))


sbp_vals = pd.to_numeric(visits_clean["sbp_mmhg"].astype(str).str.strip(), errors="coerce")
visits_clean["sbp_mmhg"] = sbp_vals.where(sbp_vals.between(70, 250)).astype("Int64")


visits_clean["weight_kg"] = pd.to_numeric(
    visits_clean["weight_kg"].astype(str).str.strip(), errors="coerce")


vd_str = visits_clean["visit_date"].fillna("").astype(str)
is_exact_ymd = vd_str.str.match(r"^\d{4}-\d{2}-\d{2}$")
visits_clean["visit_date"] = pd.to_datetime(
    vd_str.where(is_exact_ymd), format="%Y-%m-%d", errors="coerce")

visit
baseline     28496
week12       26498
BASELINE       255
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        194
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]

In [9]:
rechecks = visits_clean[visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)].sort_values(["patient_id", "visit", "visit_time"])
display(rechecks)

visits_clean = visits_clean.sort_values("visit_time").drop_duplicates(subset=["patient_id", "visit"], keep="last")

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
...,...,...,...,...,...,...,...
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56249,P28953,baseline,2026-01-21,10:30,153,74.8,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv
25464,P29170,baseline,2026-03-02,11:20,130,78.0,visits_north.csv


In [10]:
keep_cols = [
    "patient_id",
    "visit",
    "visit_date",
    "sbp_mmhg",
    "weight_kg",
    "source_file",]
visits_clean = (visits_clean.sort_values(["patient_id", "visit"])[keep_cols].reset_index(drop=True))

display(visits_clean["visit"].value_counts(dropna=False))

visits_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv",],
    "value": [
        int(len(visits_clean)),
        int((visits_clean["visit"] == "baseline").sum()),
        int((visits_clean["visit"] == "week12").sum()),
        int(visits_clean["visit_date"].isna().sum()),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        int(visits_clean["sbp_mmhg"].isna().sum()),
        int(visits_clean["sbp_mmhg"].min()),
        int(visits_clean["sbp_mmhg"].max()),
        int(visits_clean["weight_kg"].isna().sum()),
        round(float(visits_clean["weight_kg"].mean()), 2),
        int((visits_clean["source_file"] == "visits_north.csv").sum()),
        int((visits_clean["source_file"] == "visits_south.csv").sum()),],})
display(visits_summary)
visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)

visit
baseline    29599
week12      27582
Name: count, dtype: int64

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [11]:
decision_table = pd.DataFrame(
    [
        (
            "patients",
            "exact duplicate rows",
            "Drop exact duplicate rows and keep the first occurrence.",
            "Repeated export rows represent the same patient registration without new information.",
        ),
        (
            "patients",
            "gender spellings that are not a code",
            "Strip, lowercase, and map valid spellings to F, M, or X; set unknown or blank values to missing.",
            "Standardizes free-text gender entries into the three contract categories while leaving unrecorded values missing.",
        ),
        (
            "patients",
            "site codes needing normalization",
            "Uppercase site_code and remove all spaces and hyphens.",
            "Ensures consistent formatting so patient records match the normalized keys in the site directory.",
        ),
        (
            "patients",
            "birth_year sentinel 9999",
            "Replace 9999 with missing (NA) and cast to nullable Int64.",
            "9999 is a sentinel code for unrecorded birth year rather than a real birth year.",
        ),
        (
            "visits",
            "weight column named wt_kg in visits_south.csv",
            "Rename wt_kg to weight_kg before stacking with visits_north.csv.",
            "Aligns column names across both clinic exports so weights stack into a single unified column.",
        ),
        (
            "visits",
            "sbp sentinel 999 and readings outside 70 through 250",
            "Convert values outside 70-250 mmHg, including sentinel 999, to missing (NA) in Int64.",
            "Removes unmeasured sentinel codes and physiologically implausible recording errors while keeping the visit row.",
        ),
        (
            "visits",
            "impossible and loosely written visit dates",
            "Parse only exact YYYY-MM-DD calendar dates and convert invalid strings to NaT.",
            "Enforces strict date formatting and prevents non-existent calendar dates from corrupting longitudinal timelines.",
        ),
        (
            "visits",
            "repeated patient_id and visit pairs",
            "Sort by visit_time and keep the last record for each (patient_id, visit) pair.",
            "Same-visit repeats represent clinical rechecks where the later measurement is the final reading.",
        ),
        (
            "sites",
            "repeated site_code rows",
            "Filter sites to keep only rows where record_status is current.",
            "Retains the active clinic metadata per site_code and removes outdated retired versions before joining.",
        ),
    ],
    columns=["table", "issue", "decision", "reason"],
)

decision_table.to_csv(DECISION_LOG_PATH, index=False)
decision_table

,table,issue,decision,reason
0,patients,exact duplicate rows,Drop exact duplicate rows and keep the first o...,Repeated export rows represent the same patien...
1,patients,gender spellings that are not a code,"Strip, lowercase, and map valid spellings to F...",Standardizes free-text gender entries into the...
2,patients,site codes needing normalization,Uppercase site_code and remove all spaces and ...,Ensures consistent formatting so patient recor...
3,patients,birth_year sentinel 9999,Replace 9999 with missing (NA) and cast to nul...,9999 is a sentinel code for unrecorded birth y...
4,visits,weight column named wt_kg in visits_south.csv,Rename wt_kg to weight_kg before stacking with...,Aligns column names across both clinic exports...
5,visits,sbp sentinel 999 and readings outside 70 throu...,"Convert values outside 70-250 mmHg, including ...",Removes unmeasured sentinel codes and physiolo...
6,visits,impossible and loosely written visit dates,Parse only exact YYYY-MM-DD calendar dates and...,Enforces strict date formatting and prevents n...
7,visits,repeated patient_id and visit pairs,Sort by visit_time and keep the last record fo...,Same-visit repeats represent clinical rechecks...
8,sites,repeated site_code rows,Filter sites to keep only rows where record_st...,Retains the active clinic metadata per site_co...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:
a1c_long = raw["a1c_wide"].melt(
    id_vars=["patient_id"],
    value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit",
    value_name="a1c_pct",)

a1c_long["visit"] = a1c_long["visit"].map(
    {"a1c_baseline": "baseline", "a1c_week12": "week12"})

a1c_long["a1c_pct"] = a1c_long["a1c_pct"].astype("float64")
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)

a1c_summary = pd.DataFrame([
    {
        "visit": v,
        "results": int((a1c_long["visit"] == v).sum()),
        "a1c_pct_mean": round(
            float(a1c_long.loc[a1c_long["visit"] == v, "a1c_pct"].mean()), 2),}
    for v in ["baseline", "week12"]])

display(a1c_summary)
a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)

,visit,results,a1c_pct_mean
0,baseline,29107,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [13]:
sites = raw["sites"]

try:
    pd.merge(
        patients_clean,
        sites,
        on="site_code",
        how="left",
        validate="many_to_one",)
except pd.errors.MergeError as err:
    print("MergeError:", err)


display(sites[sites.duplicated(subset=["site_code"], keep=False)])

sites_current = (
    sites.loc[
        sites["record_status"] == "current", ["site_code", "site_name"]]
    .copy(deep=True)
    .reset_index(drop=True))
display(sites_current)

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


,site_code,site_name
0,N01,North Clinic
1,S02,South Valley Clinic
2,W03,West Clinic


In [14]:
visit_a1c_outer = pd.merge(
    visits_clean,
    a1c_long,
    on=["patient_id", "visit"],
    how="outer",
    indicator=True,
)
merge_counts = visit_a1c_outer["_merge"].value_counts()
display(merge_counts)

merge_audit = pd.DataFrame(
    [
        ("visits with an A1c result", int((visit_a1c_outer["_merge"] == "both").sum())),
        ("visits without an A1c result", int((visit_a1c_outer["_merge"] == "left_only").sum())),
        ("A1c results without a visit", int((visit_a1c_outer["_merge"] == "right_only").sum())),
        (
            "visits whose patient is not in the registry",
            int((~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum()),
        ),
        (
            "registry patients with no visit",
            int((~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum()),
        ),
        (
            "registry patients whose site is not in the site list",
            int((~patients_clean["site_code"].isin(sites["site_code"])).sum()),
        ),
        (
            "site list rows removed by the current-record rule",
            int(len(sites) - len(sites_current)),
        ),
    ],
    columns=["check", "count"],)

display(merge_audit)
merge_audit.to_csv(MERGE_AUDIT_PATH, index=False)

_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64

,check,count
0,visits with an A1c result,54006
1,visits without an A1c result,3175
2,A1c results without a visit,2754
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,555
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [15]:
sbp_wide = visits_clean.pivot(
    index="patient_id", columns="visit", values="sbp_mmhg").reset_index()
sbp_wide.columns.name = None

sbp_wide = sbp_wide[["patient_id", "baseline", "week12"]]
sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]

display(sbp_wide.head())

sbp_wide_sample = sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)]
display(sbp_wide_sample)
sbp_wide_sample.to_csv(SBP_WIDE_SAMPLE_PATH, index=False)

,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
1,P00002,132,129,-3
2,P00003,148,137,-11
3,P00004,142,<NA>,<NA>
4,P00005,139,127,-12


,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
3,P00004,142,<NA>,<NA>
87,P00088,152,147,-5
259,P00260,70,138,68
445,P00446,250,144,-106
486,P00487,<NA>,143,<NA>
849,P00851,<NA>,129,<NA>
2336,P02354,127,125,-2
3789,P03836,134,125,-9
5684,P05763,161,151,-10


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [16]:
complete_pairs = sbp_wide[["baseline", "week12"]].dropna()

pairs_array = np.array(complete_pairs, dtype=float)

print("complete_pairs shape:", pairs_array.shape)
visit_means = pairs_array.mean(axis=0)
print("Per-visit means (baseline, week12):", visit_means)

complete_pairs shape: (27149, 2)
Per-visit means (baseline, week12): [143.01337066 137.15061328]


### 4.4 Join the analysis table

In [17]:
analysis = (visits_clean.merge(
        patients_clean, on="patient_id", how="left", validate="many_to_one"
    )
    .merge(sites_current, on="site_code", how="left", validate="many_to_one")
    .merge(
        a1c_long, on=["patient_id", "visit"], how="left", validate="one_to_one"))

ordered_cols = [
    "patient_id",
    "visit",
    "visit_date",
    "sbp_mmhg",
    "weight_kg",
    "a1c_pct",
    "gender",
    "birth_year",
    "site_code",
    "site_name",
    "source_file",]

analysis = analysis[ordered_cols].astype({
    "patient_id": "string",
    "visit": "string",
    "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string",})

print("analysis shape:", analysis.shape)
print(analysis.dtypes)
display(analysis.head())

analysis shape: (57181, 11)
patient_id             string
visit                  string
visit_date     datetime64[us]
sbp_mmhg                Int64
weight_kg             float64
a1c_pct               float64
gender                 string
birth_year              Int64
site_code              string
site_name              string
source_file            string
dtype: object


,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00002,baseline,2026-02-20,132,85.2,8.4,F,<NA>,N01,North Clinic,visits_north.csv
3,P00002,week12,2026-05-08,129,82.0,7.4,F,<NA>,N01,North Clinic,visits_north.csv
4,P00003,baseline,2026-01-16,148,67.5,5.9,F,1970,N01,North Clinic,visits_north.csv


## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [18]:
checks = pd.Series({
    "analysis rows equal visits_clean rows": len(analysis) == len(visits_clean),
    "unique patient_id and visit pairs": (
        analysis.duplicated(subset=["patient_id", "visit"]).sum() == 0
    ),
    "sbp_mmhg within 70 to 250": (
        bool(analysis["sbp_mmhg"].dropna().between(70, 250).all())
    ),
    "site_name in sites_current": (
        bool(
            analysis["site_name"]
            .dropna()
            .isin(sites_current["site_name"])
            .all()
        )),
    "sbp_wide patient_id is unique": bool(sbp_wide["patient_id"].is_unique),
    "raw tables unchanged from snapshot": all(
        raw[name].equals(raw_snapshot[name]) for name in raw),})

display(checks)
assert checks.all(), checks[~checks]

analysis rows equal visits_clean rows    True
unique patient_id and visit pairs        True
sbp_mmhg within 70 to 250                True
site_name in sites_current               True
sbp_wide patient_id is unique            True
raw tables unchanged from snapshot       True
dtype: bool

### 5.2 Save the analysis extract

In [19]:
analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
print("analysis_sample shape:", analysis_sample.shape)
display(analysis_sample.head(10))

analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)

analysis_sample shape: (28, 11)


,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00004,baseline,2026-02-25,142,106.2,7.3,X,<NA>,N01,North Clinic,visits_north.csv
3,P00088,baseline,2026-03-10,152,68.2,8.4,F,1980,N01,North Clinic,visits_north.csv
4,P00088,week12,2026-06-03,147,66.3,7.6,F,1980,N01,North Clinic,visits_north.csv
5,P00260,baseline,NaT,70,80.4,9.5,F,1979,E04,<NA>,visits_south.csv
6,P00260,week12,2026-06-10,138,78.6,8.0,F,1979,E04,<NA>,visits_south.csv
7,P00446,baseline,NaT,250,NaN,6.5,F,1946,S02,South Valley Clinic,visits_south.csv
8,P00446,week12,2026-04-02,144,94.3,6.8,F,1946,S02,South Valley Clinic,visits_south.csv
9,P00487,baseline,2026-03-02,<NA>,103.0,7.9,M,1959,S02,South Valley Clinic,visits_south.csv


### 5.3 Read the file back

In [20]:
sample_dtypes = {
    "patient_id": "string",
    "visit": "string",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string",}

round_trip = pd.read_csv(
    ANALYSIS_SAMPLE_PATH,
    dtype=sample_dtypes,
    parse_dates=["visit_date"],)
round_trip["visit_date"] = round_trip["visit_date"].astype("datetime64[us]")

print("round_trip equals analysis_sample:", round_trip.equals(analysis_sample))

round_trip equals analysis_sample: True
